In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [22]:
train_df = pd.read_csv("trainingData.xls")

train_df.head()

test_df = pd.read_csv("validationData.xls")

test_df.head()

,WAP001,WAP002,WAP003,WAP004,WAP005,WAP006,WAP007,WAP008,WAP009,WAP010,...,WAP520,LONGITUDE,LATITUDE,FLOOR,BUILDINGID,SPACEID,RELATIVEPOSITION,USERID,PHONEID,TIMESTAMP
0,100,100,100,100,100,100,100,100,100,100,...,100,-7515.916799,4.864890e+06,1,1,0,0,0,0,1380872703
1,100,100,100,100,100,100,100,100,100,100,...,100,-7383.867221,4.864840e+06,4,2,0,0,0,13,1381155054
2,100,100,100,100,100,100,100,100,100,100,...,100,-7374.302080,4.864847e+06,4,2,0,0,0,13,1381155095
3,100,100,100,100,100,100,100,100,100,100,...,100,-7365.824883,4.864843e+06,4,2,0,0,0,13,1381155138
4,100,100,100,100,100,100,100,100,100,100,...,100,-7641.499303,4.864922e+06,2,0,0,0,0,2,1380877774


In [23]:
#Cleaning

# Training Data

# 1. Handling missing RSSI values

# Replace 100 with -110 (weak signal)
train_df.iloc[:, :520] = train_df.iloc[:, :520].replace(100, -110)

# 2. Separating  features & targets

X_train = train_df.iloc[:, :520]
y_class_train = train_df[['BUILDINGID', 'FLOOR']]
y_reg_train = train_df[['LONGITUDE', 'LATITUDE']]


In [24]:
#Cleaning

# TEST DATA

# 1. Handling missing RSSI values

test_df.iloc[:, :520] = test_df.iloc[:, :520].replace(100, -110)

# 2. Separating  features & targets

X_test = test_df.iloc[:, :520]
y_class_test = test_df[['BUILDINGID', 'FLOOR']]
y_reg_test = test_df[['LONGITUDE', 'LATITUDE']]




In [25]:
# Scaling 
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)  # ONLY transform, not fit

y_reg_train_scaled = scaler.fit_transform(y_reg_train)
y_reg_test_scaled = scaler.transform(y_reg_test)



In [26]:
#Traning models

In [42]:
#1) MLP Regressor (Regression)
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

mlp = MLPRegressor(
    hidden_layer_sizes=(100,50),
    max_iter=500,
    early_stopping=True,
    learning_rate='adaptive',
    n_iter_no_change=20,
    random_state=42
)

# Train
mlp.fit(X_train, y_reg_train_scaled)

# Predict
y_pred_scaled = mlp.predict(X_test)
y_pred_mlp = scaler.inverse_transform(y_pred_scaled)

# Evaluate
rmse = np.sqrt(mean_squared_error(y_reg_test, y_pred_mlp))
mae = mean_absolute_error(y_reg_test, y_pred_mlp)
r2 = r2_score(y_reg_test,y_pred_mlp )


print("MLP RMSE:", rmse)
print("MLP MAE:", mae)
print("MLP r^2:", r2)


MLP RMSE: 17.121111062864294
MLP MAE: 9.862963088656112
MLP r^2: 0.9667243093583255


In [29]:
 #2) KNN (Classification)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, make_scorer
import numpy as np

# =====================================================
# Custom scorer → fixes "multiclass-multioutput" warning
# =====================================================
def multioutput_accuracy(y_true, y_pred):
    return np.mean(np.all(y_true == y_pred, axis=1))

custom_scorer = make_scorer(multioutput_accuracy)

# =====================================================
# Grid Search for best K (optimization)
# =====================================================
param_grid_knn = {'n_neighbors': [3, 5, 7]}

grid_knn = GridSearchCV(
    KNeighborsClassifier(),
    param_grid_knn,
    cv=3,
    scoring=custom_scorer
)

grid_knn.fit(X_train, y_class_train)
best_knn = grid_knn.best_estimator_

print("Best K:", grid_knn.best_params_)

# =====================================================
# Predictions
# =====================================================
y_pred_knn = best_knn.predict(X_test)

building_pred = y_pred_knn[:, 0]
floor_pred = y_pred_knn[:, 1]

building_true = y_class_test['BUILDINGID'].values
floor_true = y_class_test['FLOOR'].values

# =====================================================
# BUILDING METRICS
# =====================================================
print("\n=== BUILDING METRICS ===")
print("Accuracy:", accuracy_score(building_true, building_pred))
print("Precision:", precision_score(building_true, building_pred, average='weighted'))
print("Recall:", recall_score(building_true, building_pred, average='weighted'))
print("F1 Score:", f1_score(building_true, building_pred, average='weighted'))

# =====================================================
# FLOOR METRICS
# =====================================================
print("\n=== FLOOR METRICS ===")
print("Accuracy:", accuracy_score(floor_true, floor_pred))
print("Precision:", precision_score(floor_true, floor_pred, average='weighted'))
print("Recall:", recall_score(floor_true, floor_pred, average='weighted'))
print("F1 Score:", f1_score(floor_true, floor_pred, average='weighted'))

# =====================================================
# COMBINED (Exact Location) ACCURACY
# =====================================================
combined_acc = np.mean(
    (building_pred == building_true) &
    (floor_pred == floor_true)
)

print("\n-----------------------------")
print("Combined Accuracy:", combined_acc)

Best K: {'n_neighbors': 7}

=== BUILDING METRICS ===
Accuracy: 0.990999099909991
Precision: 0.9910701229869544
Recall: 0.990999099909991
F1 Score: 0.9910118310859327

=== FLOOR METRICS ===
Accuracy: 0.8037803780378038
Precision: 0.8183665541434927
Recall: 0.8037803780378038
F1 Score: 0.8062367606525267

-----------------------------
Combined Accuracy: 0.8010801080108011


In [36]:
# 3) Decision Tree (Classification)
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

# Train model
dt_model = DecisionTreeClassifier(
    max_depth=10,
    random_state=42
)

dt_model.fit(X_train, y_class_train)

# Predict
dt_pred = dt_model.predict(X_test)

# Separate predictions
building_pred = dt_pred[:, 0]
floor_pred = dt_pred[:, 1]

# True values
building_true = y_class_test['BUILDINGID']
floor_true = y_class_test['FLOOR']

# =====================================================
# BUILDING METRICS
# =====================================================

print("=== BUILDING METRICS ===")
print("Accuracy:", accuracy_score(building_true, building_pred))
print("Precision:", precision_score(building_true, building_pred, average='weighted'))
print("Recall:", recall_score(building_true, building_pred, average='weighted'))
print("F1 Score:", f1_score(building_true, building_pred, average='weighted'))

# =====================================================
# FLOOR METRICS
# =====================================================

print("\n=== FLOOR METRICS ===")
print("Accuracy:", accuracy_score(floor_true, floor_pred))
print("Precision:", precision_score(floor_true, floor_pred, average='weighted'))
print("Recall:", recall_score(floor_true, floor_pred, average='weighted'))
print("F1 Score:", f1_score(floor_true, floor_pred, average='weighted'))

# =====================================================
# COMBINED (Exact Location) ACCURACY
# =====================================================

combined_acc = np.mean(
    (building_pred == building_true) &
    (floor_pred == floor_true)
)

print("\n-----------------------------")
print("Combined Accuracy:", combined_acc)

=== BUILDING METRICS ===
Accuracy: 0.9261926192619262
Precision: 0.9405856024964862
Recall: 0.9261926192619262
F1 Score: 0.9284685236942005

=== FLOOR METRICS ===
Accuracy: 0.5958595859585959
Precision: 0.6710328464304762
Recall: 0.5958595859585959
F1 Score: 0.6033045650428497

-----------------------------
Combined Accuracy: 0.5760576057605761


In [37]:
#4 Random Forest (Classification)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

# Train model
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_class_train)

# Predict
rf_pred = rf_model.predict(X_test)

# Separate predictions
building_pred = rf_pred[:, 0]
floor_pred = rf_pred[:, 1]

# True values
building_true = y_class_test['BUILDINGID']
floor_true = y_class_test['FLOOR']

# =====================================================
# BUILDING METRICS
# =====================================================

print("=== BUILDING METRICS ===")
print("Accuracy:", accuracy_score(building_true, building_pred))
print("Precision:", precision_score(building_true, building_pred, average='weighted'))
print("Recall:", recall_score(building_true, building_pred, average='weighted'))
print("F1 Score:", f1_score(building_true, building_pred, average='weighted'))

# =====================================================
# FLOOR METRICS
# =====================================================

print("\n=== FLOOR METRICS ===")
print("Accuracy:", accuracy_score(floor_true, floor_pred))
print("Precision:", precision_score(floor_true, floor_pred, average='weighted'))
print("Recall:", recall_score(floor_true, floor_pred, average='weighted'))
print("F1 Score:", f1_score(floor_true, floor_pred, average='weighted'))

# =====================================================
# COMBINED (Exact Location) ACCURACY
# =====================================================

combined_acc = np.mean(
    (building_pred == building_true) &
    (floor_pred == floor_true)
)

print("\n-----------------------------")
print("Combined Accuracy:", combined_acc)

=== BUILDING METRICS ===
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0

=== FLOOR METRICS ===
Accuracy: 0.9063906390639064
Precision: 0.9121278158928516
Recall: 0.9063906390639064
F1 Score: 0.9062626322929156

-----------------------------
Combined Accuracy: 0.9063906390639064


In [43]:
#5 Gradient Boosting (Regression)
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

gbr = MultiOutputRegressor(
    HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=5,
        max_iter=300,
        random_state=42
    )
)

# Train
gbr.fit(X_train, y_reg_train)

# Predict
gbr_pred = gbr.predict(X_test)

# Evaluate
rmse = np.sqrt(mean_squared_error(y_reg_test, gbr_pred))
mae = mean_absolute_error(y_reg_test, gbr_pred)
r2 = r2_score(y_reg_test, gbr_pred)

print("Gradient Boosting RMSE:", rmse)
print("Gradient Boosting MAE:", mae)
print("Gradient Boosting R²:", r2)


Gradient Boosting RMSE: 22.059235934522658
Gradient Boosting MAE: 14.132040055537336
Gradient Boosting R²: 0.9471248010083583


In [44]:
# 6) Linear Regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

lr_model = LinearRegression()

# Train
lr_model.fit(X_train, y_reg_train)

# Predict
lr_pred = lr_model.predict(X_test)

# Evaluate
rmse = np.sqrt(mean_squared_error(y_reg_test, lr_pred))
mae = mean_absolute_error(y_reg_test, lr_pred)
r2 = r2_score(y_reg_test, lr_pred)

print("Linear Regression RMSE:", rmse)
print("Linear Regression MAE:", mae)
print("Linear Regression R²:", r2)

Linear Regression RMSE: 47.37784463054917
Linear Regression MAE: 32.69891092508858
Linear Regression R²: 0.7722648249213266
